In [1]:
!nvidia-smi
import torch
print(torch.__version__, torch.cuda.is_available())

Sat Oct 31 14:41:18 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   57C    P8             14W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
%cd /content
!rm -rf sketch2shape
!git clone https://github.com/Shanksb/SKETCH2SHAPE_AI_IITB.git sketch2shape

PROJECT = "/content/sketch2shape/sketch2shape"   # inner folder (change if P4 flattens the repo)
%cd {PROJECT}
!mkdir -p third_party outputs
!git clone https://github.com/VAST-AI-Research/TripoSR third_party/TripoSR
!ls src scripts

/content
Cloning into 'sketch2shape'...
remote: Enumerating objects: 47, done.
remote: Counting objects: 100% (47/47), done.
remote: Compressing objects: 100% (37/37), done.
remote: Total 47 (delta 18), reused 38 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (47/47), 10.82 KiB | 1.80 MiB/s, done.
Resolving deltas: 100% (18/18), done.
/content/sketch2shape/sketch2shape
Cloning into 'third_party/TripoSR'...
remote: Enumerating objects: 164, done.
remote: Counting objects: 100% (98/98), done.
remote: Compressing objects: 100% (53/53), done.
remote: Total 164 (delta 66), reused 45 (delta 45), pack-reused 66 (from 1)
Receiving objects: 100% (164/164), 36.71 MiB | 17.60 MiB/s, done.
Resolving deltas: 100% (68/68), done.
scripts:
download_weights.py  test_export.py	 test_reconstruct_errors.py
patch_triposr.py     test_preprocess.py  test_reconstruct.py

src:
export.py  preprocess.py  reconstruct.py  test_preprocess.py


In [3]:
!pip install -q omegaconf einops trimesh rembg onnxruntime xatlas moderngl "imageio[ffmpeg]" huggingface-hub scikit-image
!pip install -q transformers==4.38.2

In [4]:
%cd {PROJECT}
!python scripts/patch_triposr.py

/content/sketch2shape/sketch2shape
patched


In [5]:
%cd {PROJECT}
!python third_party/TripoSR/run.py third_party/TripoSR/examples/chair.png \
    --output-dir outputs/smoke/ --model-save-format glb

/content/sketch2shape/sketch2shape
2026-10-31 14:41:48,704 - INFO - Initializing model ...
2026-10-31 14:41:55,750 - INFO - Initializing model finished in 7046.17ms.
2026-10-31 14:41:55,751 - INFO - Processing images ...
2026-10-31 14:42:13,375 - INFO - Processing images finished in 17623.88ms.
2026-10-31 14:42:13,375 - INFO - Running image 1/1 ...
2026-10-31 14:42:13,375 - INFO - Running model ...
2026-10-31 14:42:14,576 - INFO - Running model finished in 1201.39ms.
2026-10-31 14:42:14,577 - INFO - Extracting mesh ...
2026-10-31 14:42:17,321 - INFO - Extracting mesh finished in 2744.03ms.
2026-10-31 14:42:17,321 - INFO - Exporting mesh ...
2026-10-31 14:42:17,328 - INFO - Exporting mesh finished in 7.67ms.


In [6]:
%cd {PROJECT}
!python scripts/test_reconstruct.py third_party/TripoSR/examples/chair.png 256

/content/sketch2shape/sketch2shape
model load: 8.8s
reconstruct @res 256: 3.9s
peak VRAM: 2.50 GB
vertices: 42078, faces: 84160
bounds extents: [0.56859523 1.06613371 0.60370147]
saved outputs/test_mesh.glb


In [7]:
%cd {PROJECT}
for res in [128, 256, 320]:
    print(f"----- resolution {res} -----")
    !python scripts/test_reconstruct.py third_party/TripoSR/examples/chair.png {res}

/content/sketch2shape/sketch2shape
----- resolution 128 -----
model load: 8.0s
reconstruct @res 128: 1.6s
peak VRAM: 1.96 GB
vertices: 10210, faces: 20420
bounds extents: [0.57272381 1.06912285 0.61036038]
saved outputs/test_mesh.glb
----- resolution 256 -----
model load: 7.3s
reconstruct @res 256: 4.2s
peak VRAM: 2.50 GB
vertices: 42078, faces: 84160
bounds extents: [0.56859523 1.06613371 0.60370147]
saved outputs/test_mesh.glb
----- resolution 320 -----
model load: 8.5s
reconstruct @res 320: 6.3s
peak VRAM: 3.27 GB
vertices: 66388, faces: 132784
bounds extents: [0.566598   1.06645122 0.6044296 ]
saved outputs/test_mesh.glb


In [9]:
import os, sys
os.chdir(PROJECT)
sys.path.insert(0, PROJECT)

import trimesh, numpy as np
from PIL import Image
from rembg import remove
from src.reconstruct import reconstruct

img = remove(Image.open("third_party/TripoSR/examples/chair.png").convert("RGB"))
base = reconstruct(img)   # _orient() is already applied here

def rot(deg, axis):
    return trimesh.transformations.rotation_matrix(np.radians(deg), axis)

options = {
    "none": np.eye(4),
    "x+90": rot(90, [1, 0, 0]), "x-90": rot(-90, [1, 0, 0]),
    "z+90": rot(90, [0, 0, 1]), "z-90": rot(-90, [0, 0, 1]),
}
for name, T in options.items():
    m = base.copy()
    m.apply_transform(T)
    print(f"{name:6s} extents (x,y,z) = {np.round(m.extents, 2)}")
    m.export(f"outputs/orient_{name}.glb")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


none   extents (x,y,z) = [0.56 1.06 0.6 ]
x+90   extents (x,y,z) = [0.56 0.6  1.06]
x-90   extents (x,y,z) = [0.56 0.6  1.06]
z+90   extents (x,y,z) = [1.06 0.56 0.6 ]
z-90   extents (x,y,z) = [1.06 0.56 0.6 ]


In [ ]:
from google.colab import files
files.download(f"{PROJECT}/outputs/test_mesh.glb")
# for an orientation option, use e.g. f"{PROJECT}/outputs/orient_x+90.glb"

In [ ]:
from google.colab import files
files.download(f"{PROJECT}/outputs/orient_x+90.glb")
files.download(f"{PROJECT}/outputs/orient_x-90.glb")

In [12]:
%cd {PROJECT}
!python scripts/test_reconstruct.py third_party/TripoSR/examples/chair.png 256

/content/sketch2shape/sketch2shape
model load: 7.2s
reconstruct @res 256: 4.3s
peak VRAM: 2.50 GB
vertices: 42078, faces: 84160
bounds extents: [0.56859523 1.06613371 0.60370147]
saved outputs/test_mesh.glb


In [13]:
%cd {PROJECT}
!git pull

/content/sketch2shape/sketch2shape
Already up to date.


In [14]:
PROJECT = "/content/sketch2shape/sketch2shape"
%cd {PROJECT}
!git pull

/content/sketch2shape/sketch2shape
Already up to date.


In [20]:
PROJECT = "/content/sketch2shape/sketch2shape"
%cd {PROJECT}
!python scripts/test_reconstruct_errors.py

/content/sketch2shape/sketch2shape
[ OK ] RGB input: Input must be an RGBA image with the background removed (got mode RGB). Run prepare_for_3d() first.
[ OK ] blank RGBA: No object found in the image (background removal removed everything). Try a clearer, larger drawing.
[ OK ] tiny speck: No object found in the image (background removal removed everything). Try a clearer, larger drawing.
[ OK ] not an image: Input must be a PIL image.
[ OK ] valid chair: 42078 vertices, resolution 256, 17.68s, extents [0.57 1.07 0.6 ]


In [17]:
%cd /content
!rm -rf sketch2shape
!git clone https://github.com/Shanksb/SKETCH2SHAPE_AI_IITB.git sketch2shape
PROJECT = "/content/sketch2shape/sketch2shape"
%cd {PROJECT}
!python scripts/download_weights.py

/content
Cloning into 'sketch2shape'...
remote: Enumerating objects: 47, done.
remote: Counting objects: 100% (47/47), done.
remote: Compressing objects: 100% (37/37), done.
remote: Total 47 (delta 18), reused 38 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (47/47), 10.82 KiB | 5.41 MiB/s, done.
Resolving deltas: 100% (18/18), done.
/content/sketch2shape/sketch2shape
[1/3] Cloning TripoSR ...
Cloning into '/content/sketch2shape/sketch2shape/third_party/TripoSR'...
remote: Enumerating objects: 164, done.
remote: Counting objects: 100% (98/98), done.
remote: Compressing objects: 100% (53/53), done.
remote: Total 164 (delta 66), reused 45 (delta 45), pack-reused 66 (from 1)
Receiving objects: 100% (164/164), 36.71 MiB | 13.07 MiB/s, done.
Resolving deltas: 100% (68/68), done.
[2/3] Applying scikit-image patch ...
patched
[3/3] Downloading TripoSR weights (about 1.7 GB, first time only) ...
      ok: /root/.cache/huggingface/hub/models--stabilityai--TripoSR/snapshots/5b521936b